In [2]:
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path(
    r"C:\Users\sarun\OneDrive\Desktop\cesppl-internship"
)

predictions_path = (
    PROJECT_ROOT
    / "runs"
    / "run_04_finetuned"
    / "evaluation"
    / "predictions.csv"
)

predictions_df = pd.read_csv(predictions_path)

print("Total predictions:", len(predictions_df))
display(predictions_df.head())

Total predictions: 543


,image_path,true_index,true_class,predicted_index,predicted_class,top1_class,top1_confidence,top2_class,top2_confidence,top3_class,top3_confidence,correct
0,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,0,BIN LIFTING,0,BIN LIFTING,BIN LIFTING,99.45,SECONDARY VEHICLES,0.35,BIN WASHING,0.16,True
1,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,0,BIN LIFTING,0,BIN LIFTING,BIN LIFTING,60.72,SECONDARY VEHICLES,37.48,PRIMARY COLLECTION,1.16,True
2,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,0,BIN LIFTING,0,BIN LIFTING,BIN LIFTING,99.98,BIN WASHING,0.01,SECONDARY VEHICLES,0.01,True
3,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,0,BIN LIFTING,0,BIN LIFTING,BIN LIFTING,95.32,SECONDARY VEHICLES,2.89,BIN WASHING,1.28,True
4,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,0,BIN LIFTING,0,BIN LIFTING,BIN LIFTING,100.00,MECHANICAL SWEEPING,0.00,SECONDARY VEHICLES,0.00,True


In [4]:
misclassified_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("Misclassified images:", len(misclassified_df))

display(
    misclassified_df[
        [
            "true_class",
            "predicted_class",
            "top1_confidence",
            "image_path"
        ]
    ].head()
)

Misclassified images: 27


,true_class,predicted_class,top1_confidence,image_path
77,GATE MEETING,LFC,48.76,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...
79,GATE MEETING,MANUAL BEACH CLEANING,55.92,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...
81,GATE MEETING,MANUAL BEACH CLEANING,82.36,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...
82,GATE MEETING,ROAD SWEEPING,60.01,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...
95,GATE MEETING,PRIMARY COLLECTION,90.96,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...


In [6]:
def assign_error_category(row):
    true_class = row["true_class"]
    predicted_class = row["predicted_class"]
    confidence = row["top1_confidence"]

    visually_similar_groups = [
        {
            "ROAD SWEEPING",
            "LFC",
            "MANUAL BEACH CLEANING",
        },
        {
            "BIN LIFTING",
            "SECONDARY VEHICLES",
            "BIN WASHING",
        },
        {
            "GATE MEETING",
            "MANUAL BEACH CLEANING",
        },
    ]

    for group in visually_similar_groups:
        if (
            true_class in group
            and predicted_class in group
        ):
            return "Visually Similar Activities"

    if confidence < 60:
        return "Label Ambiguity"

    if confidence < 75:
        return "Background Interference"

    return "Needs Manual Review"


error_analysis = misclassified_df.copy()

error_analysis["Error Category"] = error_analysis.apply(
    assign_error_category,
    axis=1,
)

display(
    error_analysis[
        [
            "true_class",
            "predicted_class",
            "top1_confidence",
            "top2_class",
            "top2_confidence",
            "image_path",
            "Error Category",
        ]
    ]
)

,true_class,predicted_class,top1_confidence,top2_class,top2_confidence,image_path,Error Category
77,GATE MEETING,LFC,48.76,ROAD SWEEPING,36.82,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Label Ambiguity
79,GATE MEETING,MANUAL BEACH CLEANING,55.92,GATE MEETING,43.28,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Visually Similar Activities
81,GATE MEETING,MANUAL BEACH CLEANING,82.36,GATE MEETING,17.36,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Visually Similar Activities
82,GATE MEETING,ROAD SWEEPING,60.01,LFC,29.16,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Background Interference
95,GATE MEETING,PRIMARY COLLECTION,90.96,GATE MEETING,4.68,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Needs Manual Review
100,GATE MEETING,ROAD SWEEPING,56.28,LFC,22.50,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Label Ambiguity
129,LFC,ROAD SWEEPING,41.58,LFC,35.04,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Visually Similar Activities
131,LFC,BIN WASHING,50.30,LFC,31.12,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Label Ambiguity
142,LFC,ROAD SWEEPING,82.71,LFC,16.71,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Visually Similar Activities
145,LFC,BIN WASHING,81.52,LFC,16.59,C:/Users/sarun/OneDrive/Desktop/cesppl-interns...,Needs Manual Review


In [7]:
category_summary = (
    error_analysis["Error Category"]
    .value_counts()
    .reset_index()
)

category_summary.columns = [
    "Error Category",
    "Count",
]

display(category_summary)

,Error Category,Count
0,Visually Similar Activities,14
1,Label Ambiguity,8
2,Needs Manual Review,3
3,Background Interference,2


In [8]:
evaluation_dir = (
    PROJECT_ROOT
    / "runs"
    / "run_04_finetuned"
    / "evaluation"
)

detailed_output = (
    evaluation_dir
    / "error_categories.csv"
)

summary_output = (
    evaluation_dir
    / "error_category_summary.csv"
)

error_analysis.to_csv(
    detailed_output,
    index=False,
)

category_summary.to_csv(
    summary_output,
    index=False,
)

print("Saved detailed file:")
print(detailed_output)

print("\nSaved summary file:")
print(summary_output)

Saved detailed file:
C:\Users\sarun\OneDrive\Desktop\cesppl-internship\runs\run_04_finetuned\evaluation\error_categories.csv

Saved summary file:
C:\Users\sarun\OneDrive\Desktop\cesppl-internship\runs\run_04_finetuned\evaluation\error_category_summary.csv


## Error Analysis Summary

The misclassified validation images were reviewed using their true labels, predicted labels and top-three confidence scores.

Most errors occurred between visually similar waste-management activities. These classes often contain similar workers, vehicles, tools and backgrounds, making them difficult for the model to distinguish.

Lower-confidence mistakes suggest possible label ambiguity or background interference. A small number of high-confidence mistakes require manual review because the model may have focused on misleading visual features.

Overall, the final EfficientNetB0 model achieved strong performance, with 95.03% validation accuracy and a macro F1-score of 93.31%. The remaining errors are concentrated in a limited number of visually similar class pairs.